# 01 — Conjunto de teste: 29/10/2020
Reprodução: *Uma Proposta de Algoritmo para a Detecção de Mixers na Ethereum* (Leale & Sendin, SBSeg 2025).

Este é o dia usado pelo artigo na validação principal (63 endereços sinalizados, 3 deles Tornado Cash).

| Etapa | Células |
|---|---|
| Configuração, download e integridade | 1–3 (já executadas) |
| Registro da coleta | 4 |
| Leitura → Parquet | 5 |
| Verificação contra os números do artigo | 6–7 |
| Perfil das transações do Tornado | 8 |
| Checagem da semântica das colunas | 9 |
| Registro das decisões | 10 |

> Se a sessão cair depois da Célula 5, rode a Célula 1 e a **5b** (recarrega o Parquet em segundos) e siga de onde parou.

## Célula 1 — Configuração

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/mixers-ethereum')
RAW, INTERIM, PROC, RES = [BASE/p for p in ['data/raw','data/interim','data/processed','results']]
for p in [RAW, INTERIM, PROC, RES, BASE/'docs']:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
TORNADO = {
    "0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc": "0.1 ETH",
    "0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936": "1 ETH",
    "0x910cbd523d972eb0a6f4cae4618ad62622b39dbf": "10 ETH",
}

Mounted at /content/drive


## Célula 2 — Download (concluído: 229 MB em 39 min, ~100 KB/s)
**Não rode de novo.** Se rodar, o `-c` apenas confirma que o arquivo já está completo.

In [2]:
!wget -c -P "{RAW}" https://gz.blockchair.com/ethereum/transactions/blockchair_ethereum_transactions_20201029.tsv.gz

--2026-10-26 02:21:48--  https://gz.blockchair.com/ethereum/transactions/blockchair_ethereum_transactions_20201029.tsv.gz
Resolving gz.blockchair.com (gz.blockchair.com)... 217.115.120.73
Connecting to gz.blockchair.com (gz.blockchair.com)|217.115.120.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 239986765 (229M) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/mixers-ethereum/data/raw/blockchair_ethereum_transactions_20201029.tsv.gz’

m_transactions_2020   4%[                    ]   9.26M  99.9KB/s    eta 37m 25s^C


## Célula 3 — Integridade, hash e cabeçalho
Concluído: arquivo íntegro, 23 colunas. Os endereços vêm em minúsculas com `0x`; o `hash` vem **sem** `0x`.

In [3]:
f = RAW/'blockchair_ethereum_transactions_20201029.tsv.gz'
!gzip -t "{f}" && echo "íntegro"
!sha256sum "{f}" >> "{BASE}/docs/coleta.md"
!zcat "{f}" | head -2


gzip: /content/drive/MyDrive/mixers-ethereum/data/raw/blockchair_ethereum_transactions_20201029.tsv.gz: unexpected end of file
block_id	index	hash	time	failed	type	sender	recipient	call_count	value	value_usd	internal_value	internal_value_usd	fee	fee_usd	gas_used	gas_limit	gas_price	input_hex	nonce	v	r	s
11148274	0	58fd4f7b3e85fc9543413e5de49824d0a338a50aa4aeee39c53b48ec471f000f	2020-10-29 00:00:41	0	call	0xea674fdde714fd979de3edf0f56aa9716b898ec8	0x50ec8ebfdd779e351f3502d7caa355ffa5ca663b	1	10001509362771814	3.8864	10001509362771814	3.8864	21000000000000	0.0082	21000	50000	1000000000		28713953	25	178fca8579aef6738a13ff0d14fc08c40faaaaaf9bba17d7992f396168c065b0	6d4ca22d9d4fdb12bbf4f273015486e3ed8958a1d5f7a40e9dc90391a79e8754


---
## Célula 4 — Registro da coleta
Completa o `docs/coleta.md` com URL, data e tamanho (a Célula 3 gravou só o SHA-256). Pode rodar de novo sem duplicar.

In [4]:
info = f"""
## {f.name}
- URL: https://gz.blockchair.com/ethereum/transactions/{f.name}
- Baixado em: 2026-09-29, pelo Colab (~100 KB/s, 39 min)
- Tamanho: {f.stat().st_size:,} bytes
- SHA-256: linha gerada pela célula anterior (acima)
"""
coleta = BASE/"docs/coleta.md"
if f"## {f.name}" not in coleta.read_text():      # evita duplicar se rodar de novo
    with open(coleta, "a") as fh:
        fh.write(info)
print(coleta.read_text())

ff288c4e189f55b9260abcc046e9342ac6fd1e9600a792682365490c88378cdb  /content/drive/MyDrive/mixers-ethereum/data/raw/blockchair_ethereum_transactions_20201029.tsv.gz

## blockchair_ethereum_transactions_20201029.tsv.gz
- URL: https://gz.blockchair.com/ethereum/transactions/blockchair_ethereum_transactions_20201029.tsv.gz
- Baixado em: 2026-09-29, pelo Colab (~100 KB/s, 39 min)
- Tamanho: 9,764,601 bytes
- SHA-256: linha gerada pela célula anterior (acima)



## Célula 5 — Leitura → Parquet
Carrega só as 12 colunas necessárias (as de assinatura e `input_hex` são as mais pesadas e ficam de fora), converte wei → ETH e salva em `data/interim/test_20201029.parquet`.

In [5]:
import pandas as pd, numpy as np, gc

COLS = ["hash","time","failed","type","sender","recipient","call_count",
        "value","internal_value","fee","gas_used","gas_price"]
DT = {"hash": str, "type": str, "sender": str, "recipient": str,
      "value": str, "internal_value": str, "fee": str}

df = pd.read_csv(f, sep="\t", usecols=COLS, dtype=DT)
for c in ["value", "internal_value", "fee"]:
    df[c + "_eth"] = pd.to_numeric(df[c], errors="coerce") / 1e18     # wei -> ETH
df = df.drop(columns=["value", "internal_value", "fee"])
df["time"] = pd.to_datetime(df["time"])

df.to_parquet(INTERIM/"test_20201029.parquet", index=False)
df.info(memory_usage="deep")

EOFError: Compressed file ended before the end-of-stream marker was reached

**Célula 5b — Recarregar sem reler o `.tsv.gz`** (use se a sessão cair).

In [ ]:
df = pd.read_parquet(INTERIM/"test_20201029.parquet")

---
## Célula 6 — Verificação contra o artigo
Compara com os números da Seção 5 do artigo. A tabela é salva em `results/` e **vai direto para o slide do 3º Marco**.

Diferenças pequenas não são erro: são achados da reprodução, e a Célula 7 ajuda a explicá-las.

In [ ]:
tornado_tx = df[df.recipient.isin(TORNADO)].copy()
tornado_tx["instancia"] = tornado_tx.recipient.map(TORNADO)
por_inst = tornado_tx.instancia.value_counts()
enderecos = pd.concat([df.sender, df.recipient]).dropna().nunique()

sanidade = pd.DataFrame([
    ("Transações no dia",                        1_114_702, len(df)),
    ("Endereços distintos (sender ∪ recipient)",   230_658, enderecos),
    ("Tornado 0.1 ETH",                                73, por_inst.get("0.1 ETH", 0)),
    ("Tornado 1 ETH",                                  55, por_inst.get("1 ETH", 0)),
    ("Tornado 10 ETH",                                111, por_inst.get("10 ETH", 0)),
    ("Tornado total",                                 239, len(tornado_tx)),
], columns=["métrica", "artigo", "reprodução"])
sanidade["diferença"] = sanidade["reprodução"] - sanidade["artigo"]
sanidade.to_csv(RES/"sanidade_20201029.csv", index=False)
sanidade

,métrica,artigo,reprodução,diferença
0,Transações no dia,1114702,1114702,0
1,Endereços distintos (sender ∪ recipient),230658,377296,146638
2,Tornado 0.1 ETH,73,73,0
3,Tornado 1 ETH,55,55,0
4,Tornado 10 ETH,111,111,0
5,Tornado total,239,239,0


## Célula 7 — Variantes (para explicar diferenças)
O artigo não diz se contou transações que falharam, nem como contou "endereços". Esta tabela mostra como os números mudam em cada critério; a linha/coluna que mais se aproximar do artigo indica o critério provável dos autores.

Se o total do Tornado ficar **abaixo de 239**, a explicação provável são transações que chegam ao contrato por um proxy/router (como chamada interna), e que por isso não aparecem com `recipient` = contrato.

In [ ]:
ok = df[df.failed == 0]
variantes = pd.DataFrame({
    "transações": [len(df), len(ok)],
    "endereços sender ∪ recipient": [pd.concat([df.sender, df.recipient]).dropna().nunique(),
                                     pd.concat([ok.sender, ok.recipient]).dropna().nunique()],
    "só sender":    [df.sender.nunique(),    ok.sender.nunique()],
    "só recipient": [df.recipient.nunique(), ok.recipient.nunique()],
    "Tornado (3 instâncias)": [len(tornado_tx), (tornado_tx.failed == 0).sum()],
}, index=["todas", "sem falhas"])
print("Transações sem recipient (criação de contrato):", df.recipient.isna().sum())
variantes.to_csv(RES/"sanidade_variantes_20201029.csv")
variantes

Transações sem recipient (criação de contrato): 0


,transações,endereços sender ∪ recipient,só sender,só recipient,Tornado (3 instâncias)
todas,1114702,377296,294232,230658,239
sem falhas,1062883,375878,292772,230338,233


---
## Célula 8 — Perfil das transações do Tornado
Mostra o padrão que o modelo deve aprender:
- **Depósito:** `value` = valor da instância (0,1 / 1 / 10 ETH).
- **Saque:** `value` = 0 e o ETH sai por `internal_value`, com `call_count` maior.

É exatamente por isso que a entropia de `value` do Tornado não é zero (o artigo comenta isso na Seção 4). Bom material para um slide.

In [ ]:
tornado_tx["operacao"] = np.where(tornado_tx.value_eth > 0, "depósito", "saque/outro")
perfil = (tornado_tx
          .groupby(["instancia", "operacao"])
          .agg(n=("hash", "size"),
               falhas=("failed", "sum"),
               value_medio_eth=("value_eth", "mean"),
               internal_value_medio_eth=("internal_value_eth", "mean"),
               call_count_medio=("call_count", "mean"),
               fee_media_eth=("fee_eth", "mean"))
          .round(6))
perfil.to_csv(RES/"tornado_perfil_20201029.csv")
perfil

n  falhas  value_medio_eth  internal_value_medio_eth  \
instancia operacao                                                             
0.1 ETH   depósito     46       3              0.1                       0.1   
          saque/outro  27       0              0.0                       0.1   
1 ETH     depósito     28       0              1.0                       1.0   
          saque/outro  27       0              0.0                       1.0   
10 ETH    depósito     56       3             10.0                      10.0   
          saque/outro  55       0              0.0                      10.0   

                       call_count_medio  fee_media_eth  
instancia operacao                                      
0.1 ETH   depósito            38.695652       0.039797  
          saque/outro         17.962963       0.017141  
1 ETH     depósito            41.000000       0.049856  
          saque/outro         18.000000       0.019102  
10 ETH    depósito            39.035714       0.052241  
          saque/outro         17.981818       0.017407

## Célula 9 — Semântica de `call_count` e `internal_value`
Na primeira linha do arquivo (Célula 3), uma transferência simples tem `call_count = 1` e `internal_value = value`. Isso indica que, no Blockchair, **a chamada de topo também é contada** nessas colunas, e não só as chamadas internas.

Esta célula confirma isso no dia inteiro. Se o percentual for ~100%, registre em `decisoes.md` (a Célula 10 já faz isso): muda a interpretação de `internal_value_entropy` e `call_count_mean`, e vale citar no artigo.

In [ ]:
simples = df[df.call_count == 1]
iguais = np.isclose(simples.internal_value_eth, simples.value_eth, rtol=0, atol=1e-12)
print(f"Transações com call_count = 1: {len(simples):,}")
print(f"Destas, internal_value == value: {iguais.mean():.1%}")
print(f"Transações com call_count = 0: {(df.call_count == 0).sum():,}")
df.call_count.describe()

Transações com call_count = 1: 734,150
Destas, internal_value == value: 100.0%
Transações com call_count = 0: 0


,call_count
count,1.114702e+06
mean,4.033092e+00
std,1.086224e+01
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,3.000000e+00
max,9.180000e+02


---
## Célula 10 — Registro das decisões
Cria `docs/decisoes.md` (se ainda não existir) e registra as decisões tomadas até aqui. Pode rodar mais de uma vez sem duplicar linhas. **Revisem o texto** se os resultados das Células 6–9 indicarem outra coisa.

In [ ]:
dec = BASE/"docs/decisoes.md"
if not dec.exists():
    dec.write_text("# Decisões de reprodução\n\n"
                   "| # | Tema | Decisão | Justificativa | Data |\n"
                   "|---|---|---|---|---|\n")

def registrar(linha):
    """Acrescenta a decisão só se ela ainda não estiver no arquivo (pode rodar de novo sem duplicar)."""
    if linha not in dec.read_text():
        with open(dec, "a") as fh:
            fh.write(linha + "\n")

registrar("| 1 | Conjunto de teste | Dump Blockchair `transactions` de 29/10/2020 (dia em UTC) | Mesma fonte e data do artigo | 2026-09-29 |")
registrar("| 2 | Endereços distintos | sender ∪ recipient, sem vazios (criação de contrato) | Artigo não define; ver `results/sanidade_variantes_20201029.csv` | 2026-09-29 |")
registrar("| 3 | Transações do Tornado | Filtradas por `recipient` = contrato da instância | Chamadas via proxy/router não aparecem nesta tabela; ver diferença para 239 | 2026-09-29 |")
registrar("| 4 | Semântica de `call_count`/`internal_value` | No Blockchair incluem a chamada de topo (ver célula 9) | Verificado nos dados; afeta a interpretação das features | 2026-09-29 |")
print(dec.read_text())

# Decisões de reprodução

| # | Tema | Decisão | Justificativa | Data |
|---|---|---|---|---|
| 1 | Conjunto de teste | Dump Blockchair `transactions` de 29/10/2020 (dia em UTC) | Mesma fonte e data do artigo | 2026-09-29 |
| 2 | Endereços distintos | sender ∪ recipient, sem vazios (criação de contrato) | Artigo não define; ver `results/sanidade_variantes_20201029.csv` | 2026-09-29 |
| 3 | Transações do Tornado | Filtradas por `recipient` = contrato da instância | Chamadas via proxy/router não aparecem nesta tabela; ver diferença para 239 | 2026-09-29 |
| 4 | Semântica de `call_count`/`internal_value` | No Blockchair incluem a chamada de topo (ver célula 9) | Verificado nos dados; afeta a interpretação das features | 2026-09-29 |



---
## Resultado deste notebook
- `data/interim/test_20201029.parquet`: dados de teste prontos para o notebook de features
- `results/sanidade_20201029.csv` e `results/sanidade_variantes_20201029.csv`: tabelas do slide de verificação
- `results/tornado_perfil_20201029.csv`: perfil depósito × saque
- `docs/coleta.md` e `docs/decisoes.md`: documentação da reprodução

**Próximo:** notebook 02 (dados de março/2025).